# GeoRoad Inspector — YOLOv8 Training on RDD2022

Fine-tunes YOLOv8s on RDD2022 for 4 classes: Longitudinal Crack, Transverse Crack, Alligator Crack, Pothole.

**Before running:** Runtime → Change runtime type → T4 GPU

Expected time: ~15–25 min on T4 GPU.

## Step 1 — Verify GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

## Step 2 — Install dependencies

In [ ]:
!pip install ultralytics requests tqdm Pillow -q
from ultralytics import YOLO
import ultralytics
print('Ultralytics:', ultralytics.__version__)

## Step 3 — Download RDD2022 dataset

In [ ]:
import os, zipfile, requests
from tqdm import tqdm

os.makedirs('/content/rdd2022', exist_ok=True)

COUNTRY_URLS = {
    'Japan':         'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_Japan.zip',
    'India':         'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_India.zip',
    'Czech':         'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_Czech.zip',
    'United_States': 'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_United_States.zip',
}

def download_file(url, dest):
    if os.path.exists(dest):
        print(f'Already downloaded: {dest}')
        return
    print(f'Downloading {os.path.basename(dest)}...')
    r = requests.get(url, stream=True)
    total = int(r.headers.get('content-length', 0))
    with open(dest, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
        for chunk in r.iter_content(8192):
            f.write(chunk)
            bar.update(len(chunk))

for country, url in COUNTRY_URLS.items():
    zip_path = f'/content/rdd2022/{country}.zip'
    download_file(url, zip_path)
    print(f'Extracting {country}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/rdd2022/')
    print(f'  Done.')

print('\nAll datasets extracted.')

## Step 4 — Probe directory structure

This cell auto-detects the actual directory layout inside the zips before conversion.

In [ ]:
import glob

# Print top-level structure for one country to detect real paths
base = '/content/rdd2022'
print('Top-level directories:')
for d in sorted(os.listdir(base)):
    full = os.path.join(base, d)
    if os.path.isdir(full):
        print(f'  {d}/')
        for sub in sorted(os.listdir(full))[:5]:
            print(f'    {sub}/')
            inner = os.path.join(full, sub)
            if os.path.isdir(inner):
                for subsub in sorted(os.listdir(inner))[:5]:
                    print(f'      {subsub}/')

# Find all jpg files under rdd2022 and show sample paths
all_jpgs = glob.glob('/content/rdd2022/**/*.jpg', recursive=True)
print(f'\nTotal .jpg files found: {len(all_jpgs)}')
print('Sample paths:')
for p in all_jpgs[:5]:
    print(' ', p)

## Step 5 — Convert Pascal VOC → YOLO format

Auto-detects image and annotation directories per country.

In [ ]:
import glob, shutil, random
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image as PILImage

CLASS_MAP = {'D00': 0, 'D10': 1, 'D20': 2, 'D40': 3}

for split in ['train', 'val']:
    os.makedirs(f'/content/dataset/images/{split}', exist_ok=True)
    os.makedirs(f'/content/dataset/labels/{split}', exist_ok=True)

def convert_voc_to_yolo(xml_path, img_w, img_h):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        if name not in CLASS_MAP:
            continue
        cls = CLASS_MAP[name]
        bb = obj.find('bndbox')
        xmin = float(bb.find('xmin').text)
        ymin = float(bb.find('ymin').text)
        xmax = float(bb.find('xmax').text)
        ymax = float(bb.find('ymax').text)
        cx = (xmin + xmax) / 2 / img_w
        cy = (ymin + ymax) / 2 / img_h
        w  = (xmax - xmin) / img_w
        h  = (ymax - ymin) / img_h
        lines.append(f'{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    return lines

def find_dir(base, *candidates):
    """Return first existing directory from candidates."""
    for c in candidates:
        p = os.path.join(base, c)
        if os.path.isdir(p):
            return p
    return None

all_samples = []

for country in ['Japan', 'India', 'Czech', 'United_States']:
    country_base = os.path.join('/content/rdd2022', f'RDD2022_{country}')
    if not os.path.exists(country_base):
        # Try alternate naming
        country_base = os.path.join('/content/rdd2022', country)
    if not os.path.exists(country_base):
        print(f'  Skipping {country} — not found')
        continue

    # Detect image directory
    img_dir = find_dir(country_base,
        'images/train',
        'train/images',
        'images',
        'train',
        'JPEGImages',
    )
    # Detect annotation directory
    ann_dir = find_dir(country_base,
        'annotations/xmls',
        'annotations/train',
        'train/annotations',
        'annotations',
        'Annotations',
        'xmls',
    )

    if not img_dir or not ann_dir:
        # Fallback: search recursively
        jpgs = glob.glob(f'{country_base}/**/*.jpg', recursive=True)
        xmls = glob.glob(f'{country_base}/**/*.xml', recursive=True)
        if jpgs and xmls:
            img_dir = os.path.dirname(jpgs[0])
            ann_dir = os.path.dirname(xmls[0])
        else:
            print(f'  Skipping {country} — cannot find images or annotations')
            continue

    print(f'{country}: img_dir={img_dir}  ann_dir={ann_dir}')

    imgs = sorted(glob.glob(f'{img_dir}/*.jpg'))
    count_before = len(all_samples)
    for img_path in imgs:
        stem = Path(img_path).stem
        xml_path = os.path.join(ann_dir, f'{stem}.xml')
        if not os.path.exists(xml_path):
            # Try same-directory xml
            xml_path = os.path.join(img_dir, f'{stem}.xml')
        if not os.path.exists(xml_path):
            continue
        try:
            with PILImage.open(img_path) as im:
                w, h = im.size
        except:
            continue
        yolo_lines = convert_voc_to_yolo(xml_path, w, h)
        if not yolo_lines:
            continue
        all_samples.append((img_path, yolo_lines, stem))

    print(f'  Added {len(all_samples) - count_before} samples')

print(f'\nTotal valid samples: {len(all_samples)}')
assert len(all_samples) > 0, 'No samples found — check the probe output above'

random.seed(42)
random.shuffle(all_samples)
split_idx = int(len(all_samples) * 0.8)
train_samples = all_samples[:split_idx]
val_samples   = all_samples[split_idx:]

def write_split(samples, split_name):
    for img_path, yolo_lines, stem in samples:
        shutil.copy(img_path, f'/content/dataset/images/{split_name}/{stem}.jpg')
        with open(f'/content/dataset/labels/{split_name}/{stem}.txt', 'w') as f:
            f.write('\n'.join(yolo_lines))

write_split(train_samples, 'train')
write_split(val_samples, 'val')

print(f'Train: {len(train_samples)} images')
print(f'Val:   {len(val_samples)} images')
print(f'\nVerify:')
print(f'  images/train: {len(os.listdir("/content/dataset/images/train"))} files')
print(f'  images/val:   {len(os.listdir("/content/dataset/images/val"))} files')

## Step 6 — Create dataset YAML

In [ ]:
yaml_content = '''path: /content/dataset
train: images/train
val: images/val

nc: 4
names:
  0: longitudinal_crack
  1: transverse_crack
  2: alligator_crack
  3: pothole
'''

with open('/content/rdd2022.yaml', 'w') as f:
    f.write(yaml_content)

print('YAML written:')
print(yaml_content)

## Step 7 — Train YOLOv8s (100 epochs)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/rdd2022.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    cos_lr=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    name='road_damage',
    project='/content/runs',
    exist_ok=True,
    pretrained=True,
)

print('Training complete!')
print('Best weights: /content/runs/road_damage/weights/best.pt')

## Step 8 — Evaluate final metrics

In [ ]:
best_model = YOLO('/content/runs/road_damage/weights/best.pt')
metrics = best_model.val(data='/content/rdd2022.yaml', imgsz=640)

print('=== Final Validation Metrics ===')
print(f'Precision:  {metrics.box.mp*100:.1f}%')
print(f'Recall:     {metrics.box.mr*100:.1f}%')
print(f'mAP@50:     {metrics.box.map50*100:.1f}%')
print(f'mAP@50-95:  {metrics.box.map*100:.1f}%')
classes = ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']
print('\nPer-class mAP@50:')
for cls, ap in zip(classes, metrics.box.ap50):
    print(f'  {cls}: {ap*100:.1f}%')

## Step 9 — Download model

Downloads `road_damage_best.pt`.  
Replace `backend/models/road_damage_best.pt` in your repo with this file.

In [ ]:
import shutil
shutil.copy('/content/runs/road_damage/weights/best.pt', '/content/road_damage_best.pt')
from google.colab import files
files.download('/content/road_damage_best.pt')
print('Download started. Save as: backend/models/road_damage_best.pt')